### RQ1a description: How is each occupation's frozen task universe distributed across Automation-oriented, Augmentation-oriented, and Limited GenAI applicability mechanisms, and where do those mechanisms sit in the within-occupation importance ordering?

In [ ]:
# ------------------------------------------------------------------
# Step 1A — Load frozen inputs
# ------------------------------------------------------------------
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path(
    "/content/drive/MyDrive/"
    "GenAI-task-affordance-labor-outcomes/"
)
configuration_file = (
    BASE_DIR
    / "05_Layer3_Task_Configuration"
    / "Output"
    / "Occupational_Task_Affordance_Configuration_v7.csv"
)

exposure_file = (
    BASE_DIR
    / "07_GenAI_Exposure"
    / "Output"
    / "Occupational_Configuration_Exposure_v1.csv"
)
outcome_file = (
    BASE_DIR
    / "08_Labor_Outcomes"
    / "Output"
    / "OEWS_Occupational_Outcomes_v1.csv"
)
configuration_data = pd.read_csv(
    configuration_file,
    dtype={"soc6": "string"},
)
exposure_data = pd.read_csv(
    exposure_file,
    dtype={"soc6": "string"},
)
outcome_data = pd.read_csv(
    outcome_file,
    dtype={"soc6": "string"},
)
for df in (configuration_data, exposure_data, outcome_data):
    df["soc6"] = df["soc6"].str.strip()

In [ ]:
# ------------------------------------------------------------------
# RQ1a — Input-integrity checks
# ------------------------------------------------------------------
rq1a_mechanisms = {
    "Automation": {
        "share": "automation_share",
        "count": "automation_count",
        "position": "automation_position",
    },
    "Augmentation": {
        "share": "augmentation_share",
        "count": "augmentation_count",
        "position": "augmentation_position",
    },
    "Limited": {
        "share": "limited_share",
        "count": "limited_count",
        "position": "limited_position",
    },
}

required_columns = [
    "soc6",
    *[
        column
        for specification in rq1a_mechanisms.values()
        for column in specification.values()
    ],
]

missing_columns = sorted(
    set(required_columns) - set(configuration_data.columns)
)

if missing_columns:
    raise KeyError(
        f"RQ1a required columns are missing: {missing_columns}"
    )

rq1a_data = configuration_data[required_columns].copy()


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


# SOC6 structure
rq1a_data["soc6"] = rq1a_data["soc6"].astype("string")

require(len(rq1a_data) == 604, "Expected 604 rows.")
require(rq1a_data["soc6"].notna().all(), "Missing SOC6 values found.")
require(rq1a_data["soc6"].is_unique, "Duplicate SOC6 values found.")
require(
    rq1a_data["soc6"].eq(rq1a_data["soc6"].str.strip()).all(),
    "SOC6 values contain leading or trailing spaces.",
)
require(
    rq1a_data["soc6"].str.fullmatch(r"\d{2}-\d{4}").all(),
    "Invalid SOC6 format found.",
)

share_columns = [
    specification["share"]
    for specification in rq1a_mechanisms.values()
]

count_columns = [
    specification["count"]
    for specification in rq1a_mechanisms.values()
]

position_columns = [
    specification["position"]
    for specification in rq1a_mechanisms.values()
]

# Numeric schema
for column in share_columns + count_columns + position_columns:
    require(
        pd.api.types.is_numeric_dtype(rq1a_data[column]),
        f"{column} is not numeric.",
    )

# Mechanism shares
shares = rq1a_data[share_columns]

require(
    not shares.isna().any().any(),
    "Missing mechanism-share values found.",
)
require(
    np.isfinite(shares.to_numpy()).all(),
    "Non-finite mechanism-share values found.",
)
require(
    ((shares >= 0) & (shares <= 1)).all().all(),
    "Mechanism shares outside [0, 1] found.",
)

maximum_share_sum_deviation = (
    shares.sum(axis=1).sub(1).abs().max()
)

require(
    maximum_share_sum_deviation <= 1e-10,
    "Mechanism shares do not sum to 1 within tolerance.",
)

# Mechanism counts
counts = rq1a_data[count_columns]

require(
    not counts.isna().any().any(),
    "Missing mechanism-count values found.",
)
require(
    np.isfinite(counts.to_numpy()).all(),
    "Non-finite mechanism-count values found.",
)
require(
    counts.ge(0).all().all(),
    "Negative mechanism-count values found.",
)
require(
    np.isclose(
        counts.to_numpy(),
        np.round(counts.to_numpy()),
        atol=1e-12,
    ).all(),
    "Non-integer mechanism-count values found.",
)

# Count/share consistency and position bounds
audit_rows = []

for mechanism, specification in rq1a_mechanisms.items():
    share = rq1a_data[specification["share"]]
    count = rq1a_data[specification["count"]]
    position = rq1a_data[specification["position"]]

    zero_count = count.eq(0).to_numpy()
    zero_share = np.isclose(
        share.to_numpy(),
        0.0,
        atol=1e-12,
    )

    zero_state_mismatches = int(
        np.not_equal(zero_count, zero_share).sum()
    )

    require(
        zero_state_mismatches == 0,
        f"{mechanism}: count-zero and share-zero states disagree.",
    )

    observed_position = position.dropna()

    require(
        np.isfinite(observed_position.to_numpy()).all(),
        f"{mechanism}: non-finite observed positions found.",
    )
    require(
        observed_position.between(0, 1).all(),
        f"{mechanism}: observed positions outside [0, 1] found.",
    )

    audit_rows.append({
        "mechanism": mechanism,
        "zero_state_mismatches": zero_state_mismatches,
        "position_outside_0_1": int(
            (~observed_position.between(0, 1)).sum()
        ),
    })

rq1a_input_audit = pd.DataFrame(audit_rows)

display(pd.DataFrame([{
    "rows": len(rq1a_data),
    "unique_soc6": rq1a_data["soc6"].nunique(),
    "missing_soc6": int(rq1a_data["soc6"].isna().sum()),
    "duplicate_soc6": int(rq1a_data["soc6"].duplicated().sum()),
    "missing_share_values": int(shares.isna().sum().sum()),
    "maximum_share_sum_deviation": maximum_share_sum_deviation,
}]))

display(rq1a_input_audit)

### Mechanism composition

Composition = how much of each mechanism an occupation contains.

In [ ]:
# ------------------------------------------------------------------
# RQ1a — Mechanism-composition summary
# ------------------------------------------------------------------

composition_rows = []

for mechanism, specification in rq1a_mechanisms.items():
    share = rq1a_data[specification["share"]]

    present_n = int(share.gt(0).sum())
    absent_n = int(share.eq(0).sum())
    occupations = int(share.notna().sum())

    composition_rows.append({
        "mechanism": mechanism,
        "occupations": occupations,
        "present_n": present_n,
        "absent_n": absent_n,
        "absent_pct": 100 * absent_n / occupations,
        "mean_share": share.mean(),
        "std_share": share.std(ddof=1),
        "median_share": share.median(),
        "p25_share": share.quantile(0.25),
        "p75_share": share.quantile(0.75),
        "min_share": share.min(),
        "max_share": share.max(),
    })

rq1a_composition_summary = pd.DataFrame(composition_rows)

if not (
    rq1a_composition_summary["present_n"]
    + rq1a_composition_summary["absent_n"]
    == rq1a_composition_summary["occupations"]
).all():
    raise ValueError(
        "Present and absent counts do not reconcile."
    )

In [ ]:
# ------------------------------------------------------------------
# Table A — Mechanism composition: Results display
# ------------------------------------------------------------------

rq1a_table_a = rq1a_composition_summary.copy()

display(
    rq1a_table_a.round({
        "absent_pct": 1,
        "mean_share": 3,
        "std_share": 3,
        "median_share": 3,
        "p25_share": 3,
        "p75_share": 3,
        "min_share": 3,
        "max_share": 3,
    })
)

### Within-occupation importance position

Position = where that mechanism sits in the occupation’s importance ordering.

In [ ]:
# ------------------------------------------------------------------
# RQ1a — Within-occupation importance-position distribution
# ------------------------------------------------------------------

rq1a_position_rows = []
rq1a_missing_position_rows = []

for mechanism, columns in rq1a_mechanisms.items():

    task_count = rq1a_data[columns["count"]]
    position = rq1a_data[columns["position"]]

    mechanism_absent = task_count.eq(0)
    position_observed = task_count.gt(0) & position.notna()
    position_unavailable = task_count.gt(0) & position.isna()

    # The three states must partition all 604 occupations
    assert (
        mechanism_absent
        | position_observed
        | position_unavailable
    ).all()

    assert (
        mechanism_absent.sum()
        + position_observed.sum()
        + position_unavailable.sum()
        == 604
    )

    observed_values = position[position_observed]

    rq1a_position_rows.append({
        "mechanism": mechanism,
        "position_observed_n": position_observed.sum(),
        "mean_position": observed_values.mean(),
        "std_position": observed_values.std(),
        "median_position": observed_values.median(),
        "p25_position": observed_values.quantile(0.25),
        "p75_position": observed_values.quantile(0.75),
        "min_position": observed_values.min(),
        "max_position": observed_values.max(),
    })

    rq1a_missing_position_rows.append({
        "mechanism": mechanism,
        "mechanism_absent_n": mechanism_absent.sum(),
        "position_unavailable_n": position_unavailable.sum(),
        "position_unavailable_reason":
            "Mechanism present, but no usable Layer 3 importance position",
    })

rq1a_position_summary = pd.DataFrame(
    rq1a_position_rows
)

rq1a_missing_position_summary = pd.DataFrame(
    rq1a_missing_position_rows
)

In [ ]:
display(
    rq1a_position_summary.round({
        "mean_position": 3,
        "std_position": 3,
        "median_position": 3,
        "p25_position": 3,
        "p75_position": 3,
        "min_position": 3,
        "max_position": 3,
    })
)

In [ ]:
print("Missing-position reasons reported separately:")

with pd.option_context("display.max_colwidth", None):
    display(rq1a_missing_position_summary)

Automation 288, Augmentation 6, and Limited 6. These are not missing values. The occupation contains no tasks classified under that mechanism, so its position is not applicable.

In [ ]:
# RQ1a — Combined manuscript figure
colors = {
    "Automation": "#D55E00",
    "Augmentation": "#0072B2",
    "Limited": "#6B6B6B",
}
bins = np.linspace(0, 1, 21)

fig_combined, axes_combined = plt.subplots(
    2, 3, figsize=(7, 3.6), dpi=150, sharex="row", sharey="row",
    layout="constrained",
)
fig_combined.set_facecolor("white")

for ax in axes_combined.flat:
    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#E8EBEF", linewidth=0.7)
    ax.spines[["top", "right"]].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#C5CBD3")
        ax.spines[side].set_linewidth(0.6)
    ax.tick_params(axis="both", length=0, labelsize=6, colors="#4B5563", pad=3)
    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.margins(y=0.12)

for column, (mechanism, fields) in enumerate(rq1a_mechanisms.items()):
    ax_share, ax_position = axes_combined[:, column]
    shares = rq1a_data[fields["share"]].dropna()

    ax_share.hist(
        shares, bins=bins, color=colors[mechanism],
        edgecolor="white", linewidth=0.5, alpha=0.9,
    )
    ax_share.set_title(mechanism, fontsize=6, fontweight="normal", pad=6)
    ax_share.text(
        0.97, 0.94,
        f"Absent: {shares.eq(0).sum()} of {len(rq1a_data)}",
        transform=ax_share.transAxes, ha="right", va="top",
        fontsize=5.5, color="#4B5563",
    )

    present = rq1a_data[fields["count"]].gt(0)
    available = rq1a_data[fields["position"]].notna()
    positions = rq1a_data.loc[present & available, fields["position"]]

    ax_position.hist(
        positions, bins=bins,
        weights=np.full(len(positions), 100 / len(positions)),
        color=colors[mechanism], edgecolor="white", linewidth=0.5, alpha=0.9,
    )
    ax_position.axvline(0.5, color="#4B5563", linestyle="--", linewidth=0.9)
    ax_position.text(
        0.97, 0.94,
        f"Observed: {len(positions)}\nUnavailable: {(present & ~available).sum()}",
        transform=ax_position.transAxes, ha="right", va="top",
        fontsize=5.5, color="#4B5563",
    )

axes_combined[0, 1].set_xlabel("Share of classified tasks", fontsize=6, labelpad=5)
axes_combined[1, 1].set_xlabel("Relative importance position", fontsize=6, labelpad=5)
axes_combined[0, 0].set_ylabel("Number of occupations", fontsize=6, labelpad=5)
axes_combined[1, 0].set_ylabel(
    "Percentage of\nposition-observed occupations", fontsize=6, labelpad=5,
)
fig_combined.savefig(
    BASE_DIR / "09_RQ1_Analysis" /"Figures" / "task_share_imp_positions.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

### RQ1b incremental association: Do task-affordance configuration terms add information about employment and wage trajectories after conditioning on aggregate GenAI exposure?

Phase 1 — Sample contract

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 1A: Merge frozen DEV inputs
# ------------------------------------------------------------------
tmp_configuration = configuration_data.drop(
    columns=["human_beta"],
    errors="ignore",
)

model_data = (
    tmp_configuration
    .merge(
        exposure_data[["soc6", "human_beta"]],
        on="soc6",
        how="left",
        validate="one_to_one",
    )
    .merge(
        outcome_data,
        on="soc6",
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 1B: Construct outcome-specific common samples
# ------------------------------------------------------------------

mechanisms = ["automation", "augmentation", "limited"]

model_data["layer3_missing_reason"] = pd.Series(
    "",
    index=model_data.index,
    dtype="string",
)

for mechanism in mechanisms:
    invalid_position = (
        model_data[f"{mechanism}_count"].gt(0)
        & model_data[f"{mechanism}_position"].isna()
    )

    model_data.loc[
        invalid_position,
        "layer3_missing_reason",
    ] += f"missing_{mechanism}_position;"

model_data["layer3_missing_reason"] = (
    model_data["layer3_missing_reason"]
    .str.rstrip(";")
    .replace("", pd.NA)
)

model_data["layer3_eligible"] = (
    model_data["layer3_missing_reason"].isna()
)

outcome_columns = {
    "employment": "employment_ann_log_change",
    "mean_wage": "mean_wage_ann_log_change",
    "median_wage": "median_wage_ann_log_change",
}

composition_available = model_data[
    ["automation_share", "augmentation_share", "limited_share"]
].notna().all(axis=1)

benchmark_available = model_data["human_beta"].notna()

for outcome_name, outcome_column in outcome_columns.items():
    sample_column = f"sample_{outcome_name}"
    reason_column = f"{outcome_name}_exclusion_reason"

    outcome_available = model_data[outcome_column].notna()

    model_data[sample_column] = (
        outcome_available
        & benchmark_available
        & composition_available
        & model_data["layer3_eligible"]
    )

    model_data[reason_column] = pd.Series(
        "",
        index=model_data.index,
        dtype="string",
    )

    exclusion_conditions = {
        "missing_outcome": ~outcome_available,
        "missing_human_beta": ~benchmark_available,
        "missing_composition": ~composition_available,
        "layer3_ineligible": ~model_data["layer3_eligible"],
    }

    for reason, condition in exclusion_conditions.items():
        model_data.loc[condition, reason_column] += reason + ";"

    model_data[reason_column] = (
        model_data[reason_column]
        .str.rstrip(";")
        .replace("", pd.NA)
    )

RQ1b Phase 2 — Regressor contract: in progress

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 2: Construct frozen configuration regressors
# Exact DEV Step 3
# ------------------------------------------------------------------

# Layer 1 shares already exist in the frozen configuration
layer1_terms = [
    "automation_share",
    "augmentation_share",
]

# Structural-state indicators
model_data["automation_present"] = (
    model_data["automation_count"].gt(0).astype("int8")
)

model_data["augmentation_absent"] = (
    model_data["augmentation_count"].eq(0).astype("int8")
)

model_data["limited_absent"] = (
    model_data["limited_count"].eq(0).astype("int8")
)

# Layer 3 positions: centered at the occupational midpoint (0.5)
# and set to zero only when that mechanism is structurally absent
model_data["automation_position_centered"] = (
    model_data["automation_position"] - 0.5
).where(
    model_data["automation_present"].eq(1),
    0.0
)

model_data["augmentation_position_centered"] = (
    model_data["augmentation_position"] - 0.5
).where(
    model_data["augmentation_absent"].eq(0),
    0.0
)

model_data["limited_position_centered"] = (
    model_data["limited_position"] - 0.5
).where(
    model_data["limited_absent"].eq(0),
    0.0
)

configuration_terms = [
    *layer1_terms,
    "automation_present",
    "augmentation_absent",
    "limited_absent",
    "automation_position_centered",
    "augmentation_position_centered",
    "limited_position_centered",
]

print("model_data rows:", len(model_data))
print("Unique SOC6:", model_data["soc6"].nunique())
print("Configuration terms created or retained:")
print(configuration_terms)

In [ ]:
display(
    model_data[
        ["soc6", *configuration_terms]
    ].head()
)

Phase 3. Employment models

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 3: Estimate frozen employment models
# Exact DEV Step 5A
# ------------------------------------------------------------------

import numpy as np
import statsmodels.api as sm

employment_mask = model_data["sample_employment"]

#assert employment_mask.sum() == 540

employment_model_terms = {
    "M0": [
        "human_beta",
    ],
    "M1": [
        "human_beta",
        "automation_share",
        "augmentation_share",
    ],
    "M2": [
        "human_beta",
        "automation_share",
        "augmentation_share",
        "automation_present",
        "augmentation_absent",
        "limited_absent",
        "automation_position_centered",
        "augmentation_position_centered",
        "limited_position_centered",
    ],
}

required_columns = [
    "employment_ann_log_change",
    *employment_model_terms["M2"],
]

# The common-sample flag must guarantee complete model data
assert not model_data.loc[
    employment_mask,
    required_columns
].isna().any().any()

employment_models = {}
tmp_employment_model_summary = []

for model_name, predictors in employment_model_terms.items():

    y = model_data.loc[
        employment_mask,
        "employment_ann_log_change"
    ]

    tmp_X = sm.add_constant(
        model_data.loc[employment_mask, predictors],
        has_constant="add"
    )

    # Confirm that each design matrix is identified
    assert np.linalg.matrix_rank(tmp_X) == tmp_X.shape[1]

    employment_models[model_name] = sm.OLS(
        y,
        tmp_X
    ).fit(
        cov_type="HC3"
    )

    fitted_model = employment_models[model_name]

    tmp_employment_model_summary.append({
        "model": model_name,
        "n": int(fitted_model.nobs),
        "regressors_excluding_intercept": len(predictors),
        "r_squared": fitted_model.rsquared,
        "adjusted_r_squared": fitted_model.rsquared_adj,
    })

tmp_employment_model_summary = pd.DataFrame(
    tmp_employment_model_summary
)

In [ ]:
display(
    tmp_employment_model_summary.round({
        "r_squared": 4,
        "adjusted_r_squared": 4,
    })
)

Phase 4. Employment block tests

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 4: Prespecified employment HC3 Wald tests
# Exact DEV Step 5B
# ------------------------------------------------------------------

def hc3_wald_test(model, terms):
    parameter_names = list(model.params.index)

    restriction_matrix = np.zeros(
        (len(terms), len(parameter_names))
    )

    for row_number, term in enumerate(terms):
        restriction_matrix[
            row_number,
            parameter_names.index(term)
        ] = 1

    test_result = model.wald_test(
        restriction_matrix,
        use_f=True,
        scalar=True
    )

    return {
        "df_num": int(test_result.df_num),
        "df_denom": int(test_result.df_denom),
        "f_statistic": float(
            np.asarray(test_result.statistic).squeeze()
        ),
        "p_value": float(
            np.asarray(test_result.pvalue).squeeze()
        ),
    }

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 4: Prespecified employment HC3 Wald tests
# ------------------------------------------------------------------

employment_wald_definitions = [
    {
        "configuration_block": "Layer 1 — Mechanism composition",
        "incremental_comparison":
            "Beyond aggregate GenAI exposure (M1 vs. M0)",
        "model": "M1",
        "terms": [
            "automation_share",
            "augmentation_share",
        ],
    },
    {
        "configuration_block":
            "Layer 3 — Structural states and importance positions",
        "incremental_comparison":
            "Beyond aggregate exposure and mechanism composition "
            "(M2 vs. M1)",
        "model": "M2",
        "terms": [
            "automation_present",
            "augmentation_absent",
            "limited_absent",
            "automation_position_centered",
            "augmentation_position_centered",
            "limited_position_centered",
        ],
    },
    {
        "configuration_block":
            "Full task-affordance configuration",
        "incremental_comparison":
            "Beyond aggregate GenAI exposure (M2 vs. M0)",
        "model": "M2",
        "terms": [
            "automation_share",
            "augmentation_share",
            "automation_present",
            "augmentation_absent",
            "limited_absent",
            "automation_position_centered",
            "augmentation_position_centered",
            "limited_position_centered",
        ],
    },
    {
        "configuration_block": "Layer 3 — Importance positions",
        "incremental_comparison":
            "Conditional on exposure, composition, and structural "
            "states within M2",
        "model": "M2",
        "terms": [
            "automation_position_centered",
            "augmentation_position_centered",
            "limited_position_centered",
        ],
    },
]


rq1b_employment_wald_results = []

for specification in employment_wald_definitions:

    result = hc3_wald_test(
        employment_models[specification["model"]],
        specification["terms"],
    )

    rq1b_employment_wald_results.append({
        "configuration_block":
            specification["configuration_block"],
        "incremental_comparison":
            specification["incremental_comparison"],
        **result,
    })


rq1b_employment_wald_results = pd.DataFrame(
    rq1b_employment_wald_results
)


rq1b_employment_joint_tests = pd.DataFrame({
    "Configuration block tested":
        rq1b_employment_wald_results["configuration_block"],

    "Incremental comparison":
        rq1b_employment_wald_results["incremental_comparison"],

    "HC3-robust Wald F statistic":
        rq1b_employment_wald_results.apply(
            lambda row: (
                f"F({int(row['df_num'])}, "
                f"{int(row['df_denom'])}) = "
                f"{row['f_statistic']:.3f}"
            ),
            axis=1,
        ),

    "p value":
        rq1b_employment_wald_results["p_value"].map(
            lambda value:
                "< .001"
                if value < .001
                else f"{value:.4f}".lstrip("0")
        ),
})

In [ ]:
print("Employment Table")
print("Main Table")
display(
    rq1b_employment_joint_tests.style
    .hide(axis="index")
    .set_caption(
        "Table X. Joint Tests of Incremental Task-Affordance Configuration Terms for Annualized Log Employment Change (N = 540)"
    )
)

Phase 5 — Wage models

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 5: Mean- and median-wage models
# ------------------------------------------------------------------

wage_outcome_specifications = {
    "mean_wage": {
        "outcome": "mean_wage_ann_log_change",
        "sample_flag": "sample_mean_wage",
        "expected_n": 537,
    },
    "median_wage": {
        "outcome": "median_wage_ann_log_change",
        "sample_flag": "sample_median_wage",
        "expected_n": 526,
    },
}

wage_models = {}
tmp_wage_model_summary = []

for outcome_name, specification in wage_outcome_specifications.items():

    sample_mask = model_data[
        specification["sample_flag"]
    ]

    assert sample_mask.sum() == specification["expected_n"]

    required_columns = [
        specification["outcome"],
        *employment_model_terms["M2"],
    ]

    assert not model_data.loc[
        sample_mask,
        required_columns
    ].isna().any().any()

    wage_models[outcome_name] = {}

    for model_name, predictors in employment_model_terms.items():

        y = model_data.loc[
            sample_mask,
            specification["outcome"]
        ]

        tmp_X = sm.add_constant(
            model_data.loc[sample_mask, predictors],
            has_constant="add",
        )

        assert np.linalg.matrix_rank(tmp_X) == tmp_X.shape[1]

        fitted_model = sm.OLS(
            y,
            tmp_X,
        ).fit(
            cov_type="HC3"
        )

        wage_models[outcome_name][model_name] = fitted_model

        tmp_wage_model_summary.append({
            "outcome": outcome_name,
            "model": model_name,
            "n": int(fitted_model.nobs),
            "regressors_excluding_intercept": len(predictors),
            "r_squared": fitted_model.rsquared,
            "adjusted_r_squared": fitted_model.rsquared_adj,
        })

tmp_wage_model_summary = pd.DataFrame(
    tmp_wage_model_summary
)

In [ ]:
display(
    tmp_wage_model_summary.round({
        "r_squared": 4,
        "adjusted_r_squared": 4,
    })
)

Phase 6 — Wage HC3 Wald tests

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 6: Prespecified wage HC3 Wald tests
# ------------------------------------------------------------------

tmp_wage_wald_results = []

for outcome_name, outcome_models in wage_models.items():

    for specification in employment_wald_definitions:

        result = hc3_wald_test(
            outcome_models[specification["model"]],
            specification["terms"],
        )

        tmp_wage_wald_results.append({
            "outcome": outcome_name,
            "configuration_block":
                specification["configuration_block"],
            "incremental_comparison":
                specification["incremental_comparison"],
            "model": specification["model"],
            **result,
        })

tmp_wage_wald_results = pd.DataFrame(
    tmp_wage_wald_results
)

In [ ]:
# ------------------------------------------------------------
# Phase 6 — Publication-facing wage Wald-test tables
# Presentation only; no models are re-estimated
# ------------------------------------------------------------

def format_p_value(p):
    if p < 0.001:
        return "< .001"
    return f"{p:.4f}".lstrip("0")


def build_wage_wald_table(results, outcome):

    table = results.loc[
        results["outcome"].eq(outcome),
        [
            "configuration_block",
            "incremental_comparison",
            "df_num",
            "df_denom",
            "f_statistic",
            "p_value",
        ],
    ].copy()

    table["HC3-robust Wald F statistic"] = table.apply(
        lambda row: (
            f"F({int(row['df_num'])}, {int(row['df_denom'])}) "
            f"= {row['f_statistic']:.3f}"
        ),
        axis=1,
    )

    table["p value"] = table["p_value"].map(format_p_value)

    table = table[
        [
            "configuration_block",
            "incremental_comparison",
            "HC3-robust Wald F statistic",
            "p value",
        ]
    ]

    table.columns = [
        "Configuration block tested",
        "Incremental comparison",
        "HC3-robust Wald F statistic",
        "p value",
    ]

    return table

In [ ]:

mean_wage_wald_table = build_wage_wald_table(
    tmp_wage_wald_results,
    "mean_wage",
)

median_wage_wald_table = build_wage_wald_table(
    tmp_wage_wald_results,
    "median_wage",
)

In [ ]:
print("Mean wage")
print("Main Table")
print("Table X. HC3-Robust Wald Tests of Incremental Task-Affordance Configuration Terms for Annualized Log Mean-Wage Change")
display(mean_wage_wald_table)

In [ ]:
print("Median wage")
print("Supplement Table")
print("Table Sx. HC3-Robust Wald Tests of Incremental Task-Affordance Configuration Terms for Annualized Log Median-Wage Change")
display(median_wage_wald_table)

Phase 7 - Model-fit summaries

* M0 → M1: gain after adding Layer 1 mechanism composition
* M1 → M2: further gain after adding Layer 3 structural-state + position terms
* M0 → M2: total gain from the full configuration beyond exposur

In [ ]:
fitted_model.rsquared

In [ ]:
fitted_model.rsquared_adj

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 7A: Employment model-fit summary
# ------------------------------------------------------------------

tmp_employment_fit_summary = pd.DataFrame([
    {
        "model": model_name,
        "r_squared": fitted_model.rsquared,
        "adjusted_r_squared": fitted_model.rsquared_adj,
    }
    for model_name, fitted_model in employment_models.items()
])

assert tmp_employment_fit_summary["model"].tolist() == ["M0", "M1", "M2"]

tmp_employment_fit_summary["delta_r_squared"] = (
    tmp_employment_fit_summary["r_squared"].diff()
)

tmp_employment_fit_summary["delta_adjusted_r_squared"] = (
    tmp_employment_fit_summary["adjusted_r_squared"].diff()
)

In [ ]:
display(
    tmp_employment_fit_summary.round({
        "r_squared": 4,
        "adjusted_r_squared": 4,
        "delta_r_squared": 4,
        "delta_adjusted_r_squared": 4,
    })
)

For employment, most of the observed increase in in-sample fit occurs when Layer 1 mechanism composition is added to aggregate exposure; Layer 3 adds a smaller further increase in fit.

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 7B: Wage model-fit summaries
# ------------------------------------------------------------------

tmp_wage_fit_summary = pd.DataFrame([
    {
        "outcome": outcome_name,
        "model": model_name,
        "r_squared": fitted_model.rsquared,
        "adjusted_r_squared": fitted_model.rsquared_adj,
    }
    for outcome_name, outcome_models in wage_models.items()
    for model_name, fitted_model in outcome_models.items()
])

for outcome_name in tmp_wage_fit_summary["outcome"].unique():

    outcome_mask = tmp_wage_fit_summary["outcome"].eq(outcome_name)

    tmp_wage_fit_summary.loc[
        outcome_mask, "delta_r_squared"
    ] = (
        tmp_wage_fit_summary.loc[
            outcome_mask, "r_squared"
        ].diff()
    )

    tmp_wage_fit_summary.loc[
        outcome_mask, "delta_adjusted_r_squared"
    ] = (
        tmp_wage_fit_summary.loc[
            outcome_mask, "adjusted_r_squared"
        ].diff()
    )

display(
    tmp_wage_fit_summary.round({
        "r_squared": 4,
        "adjusted_r_squared": 4,
        "delta_r_squared": 4,
        "delta_adjusted_r_squared": 4,
    })
)

Phase 7C — Incremental employment fit display

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 7C: Incremental employment model-fit change
# Presentation only; uses Phase 7A results
# ------------------------------------------------------------------

tmp_employment_incremental_fit = (
    tmp_employment_fit_summary
    .loc[
        tmp_employment_fit_summary["model"].isin(["M1", "M2"]),
        [
            "model",
            "delta_r_squared",
            "delta_adjusted_r_squared",
        ],
    ]
    .copy()
)

tmp_employment_incremental_fit["comparison"] = (
    tmp_employment_incremental_fit["model"].map({
        "M1": "M0 → M1",
        "M2": "M1 → M2",
    })
)

tmp_employment_incremental_fit = tmp_employment_incremental_fit[
    [
        "comparison",
        "delta_r_squared",
        "delta_adjusted_r_squared",
    ]
]

display(
    tmp_employment_incremental_fit.round({
        "delta_r_squared": 4,
        "delta_adjusted_r_squared": 4,
    })
)

Phase 7D — Incremental wage fit display

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 7D: Incremental wage model-fit change
# Presentation only; uses Phase 7B results
# ------------------------------------------------------------------

tmp_wage_incremental_fit = (
    tmp_wage_fit_summary
    .loc[
        tmp_wage_fit_summary["model"].isin(["M1", "M2"]),
        [
            "outcome",
            "model",
            "delta_r_squared",
            "delta_adjusted_r_squared",
        ],
    ]
    .copy()
)

tmp_wage_incremental_fit["comparison"] = (
    tmp_wage_incremental_fit["model"].map({
        "M1": "M0 → M1",
        "M2": "M1 → M2",
    })
)

tmp_wage_incremental_fit = tmp_wage_incremental_fit[
    [
        "outcome",
        "comparison",
        "delta_r_squared",
        "delta_adjusted_r_squared",
    ]
]

display(
    tmp_wage_incremental_fit.round({
        "delta_r_squared": 4,
        "delta_adjusted_r_squared": 4,
    })
)

In [ ]:
# ------------------------------------------------------------------
# RQ1b — Phase 7C: Supplementary model-fit table
# Presentation only; uses Phase 7A and Phase 7B results
# ------------------------------------------------------------------

tmp_employment_fit_display = (
    tmp_employment_fit_summary
    .assign(outcome="Employment")
)

tmp_wage_fit_display = (
    tmp_wage_fit_summary
    .assign(
        outcome=lambda x: x["outcome"].replace({
            "mean_wage": "Mean wage",
            "median_wage": "Median wage",
        })
    )
)

tmp_model_fit_table = pd.concat(
    [
        tmp_employment_fit_display,
        tmp_wage_fit_display,
    ],
    ignore_index=True,
)

tmp_model_fit_table["specification"] = (
    tmp_model_fit_table["model"].map({
        "M0": "Aggregate GenAI exposure",
        "M1": "+ Layer 1 mechanism composition",
        "M2": "+ Layer 3 configuration",
    })
)

tmp_model_fit_table = tmp_model_fit_table[
    [
        "outcome",
        "model",
        "specification",
        "r_squared",
        "adjusted_r_squared",
        "delta_r_squared",
        "delta_adjusted_r_squared",
    ]
].rename(columns={
    "outcome": "Outcome",
    "model": "Model",
    "specification": "Comparison",
    "r_squared": "R²",
    "adjusted_r_squared": "Adjusted R²",
    "delta_r_squared": "ΔR²",
    "delta_adjusted_r_squared": "Δ Adjusted R²",
})

display(
    tmp_model_fit_table.style
    .set_caption(
        "Table SP. In-Sample Fit Across Nested RQ1b Model Specifications"
    )
    .format({
        "R²": "{:.4f}",
        "Adjusted R²": "{:.4f}",
        "ΔR² from prior model": lambda x: "—" if pd.isna(x) else f"{x:.4f}",
        "Δ Adjusted R² from prior model":
            lambda x: "—" if pd.isna(x) else f"{x:.4f}",
    })
    .hide(axis="index")
)

### RQ1c. Do task-affordance configuration–outcome associations differ across Education, Finance, Healthcare, and Legal Services?

### Sector heterogeneity

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 1: Outcome-specific sector-membership audit
# Exact DEV Step 7A
# ------------------------------------------------------------------

sector_cols = {
    "finance": "sector_finance",
    "healthcare": "sector_healthcare",
    "education": "sector_education",
    "legal": "sector_legal",
}

outcome_flags = {
    "employment": "sample_employment",
    "mean_wage": "sample_mean_wage",
    "median_wage": "sample_median_wage",
}

audit_rows = []

for outcome, sample_flag in outcome_flags.items():

    tmp_sector = model_data.loc[
        model_data[sample_flag],
        ["soc6", *sector_cols.values()]
    ].copy()

    tmp_sector["sector_memberships"] = (
        tmp_sector[list(sector_cols.values())]
        .astype(int)
        .sum(axis=1)
    )

    audit_rows.append({
        "outcome": outcome,
        "common_sample_occupations": len(tmp_sector),
        "stacked_rows": int(tmp_sector["sector_memberships"].sum()),
        "multiple_sector_occupations":
            int((tmp_sector["sector_memberships"] > 1).sum()),
        "no_sector_occupations":
            int((tmp_sector["sector_memberships"] == 0).sum()),
        **{
            sector: int(tmp_sector[column].sum())
            for sector, column in sector_cols.items()
        },
    })

tmp_sector_audit = pd.DataFrame(audit_rows)

In [ ]:
display(tmp_sector_audit)

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 2: Construct the temporary employment sector stack
# Exact DEV Step 7B
# ------------------------------------------------------------------

sector_model_terms = [
    "human_beta",
    "automation_share",
    "augmentation_share",
    "automation_present",
    "augmentation_absent",
    "limited_absent",
    "automation_position_centered",
    "augmentation_position_centered",
    "limited_position_centered",
]

tmp_employment_base = model_data.loc[
    model_data["sample_employment"],
    [
        "soc6",
        "employment_ann_log_change",
        *sector_model_terms,
        *sector_cols.values(),
    ]
].copy()

tmp_sector_employment = tmp_employment_base.melt(
    id_vars=[
        "soc6",
        "employment_ann_log_change",
        *sector_model_terms,
    ],
    value_vars=list(sector_cols.values()),
    var_name="sector_column",
    value_name="sector_member",
)

tmp_sector_employment = (
    tmp_sector_employment.loc[
        tmp_sector_employment["sector_member"]
    ]
    .drop(columns="sector_member")
    .assign(
        sector=lambda df:
            df["sector_column"]
            .str.replace("sector_", "", regex=False)
            .str.title()
    )
    .drop(columns="sector_column")
    .reset_index(drop=True)
)

assert len(tmp_sector_employment) == 1226
assert tmp_sector_employment["soc6"].nunique() == 540
assert not tmp_sector_employment.duplicated(
    ["soc6", "sector"]
).any()

display(
    tmp_sector_employment.groupby("sector")
    .agg(
        stacked_rows=("soc6", "size"),
        unique_occupations=("soc6", "nunique"),
    )
    .reset_index()
)

print("Stacked rows:", len(tmp_sector_employment))
print(
    "Unique SOC6 clusters:",
    tmp_sector_employment["soc6"].nunique()
)

Employment sector heterogeneity model.

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 3: Fit employment sector model
# Final corrected DEV Step 7C
# ------------------------------------------------------------------

import statsmodels.formula.api as smf

estimable_sector_terms = [
    term for term in sector_model_terms
    if term != "limited_absent"
]

# Limited-absence effects only for sectors where variation exists
for sector in ["Education", "Finance", "Healthcare"]:

    column = f"limited_absent_{sector.lower()}"

    tmp_sector_employment[column] = (
        (tmp_sector_employment["sector"] == sector)
        & tmp_sector_employment["limited_absent"].astype(bool)
    ).astype(int)

limited_absent_terms = [
    "limited_absent_education",
    "limited_absent_finance",
    "limited_absent_healthcare",
]

employment_sector_formula = (
    "employment_ann_log_change ~ 0 + C(sector) + "
    + " + ".join(
        f"C(sector):{term}"
        for term in estimable_sector_terms
    )
    + " + "
    + " + ".join(limited_absent_terms)
)

employment_sector_model = smf.ols(
    formula=employment_sector_formula,
    data=tmp_sector_employment,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": tmp_sector_employment["soc6"],
        "use_correction": True,
    },
    use_t=True,
)

tmp_employment_sector_design_audit = pd.DataFrame({
    "stacked_rows": [int(employment_sector_model.nobs)],
    "soc6_clusters": [
        tmp_sector_employment["soc6"].nunique()
    ],
    "estimated_parameters": [
        len(employment_sector_model.params)
    ],
    "design_matrix_rank": [
        np.linalg.matrix_rank(
            employment_sector_model.model.exog
        )
    ],
})

In [ ]:
display(tmp_employment_sector_design_audit)

assert len(employment_sector_model.params) == 39
assert np.linalg.matrix_rank(
    employment_sector_model.model.exog
) == 39

Employment cross-sector equality tests

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 4: Employment cross-sector equality tests
# Exact DEV Step 7D + reader-facing presentation
# ------------------------------------------------------------------

comparison_sectors = ["Education", "Finance", "Healthcare"]
reference_sector = "Legal"


def sector_equalities(terms):
    return [
        (
            f"C(sector)[{sector}]:{term} = "
            f"C(sector)[{reference_sector}]:{term}"
        )
        for term in terms
        for sector in comparison_sectors
    ]


layer1_equality = sector_equalities([
    "automation_share",
    "augmentation_share",
])

position_equality = sector_equalities([
    "automation_position_centered",
    "augmentation_position_centered",
    "limited_position_centered",
])

layer3_equality = (
    sector_equalities([
        "automation_present",
        "augmentation_absent",
        "automation_position_centered",
        "augmentation_position_centered",
        "limited_position_centered",
    ])
    + [
        "limited_absent_finance = limited_absent_education",
        "limited_absent_healthcare = limited_absent_education",
    ]
)

full_configuration_equality = (
    layer1_equality + layer3_equality
)


equality_tests = {
    "Layer 1 composition": layer1_equality,
    "Total Layer 3": layer3_equality,
    "Full configuration": full_configuration_equality,
    "Headline Layer 3 position": position_equality,
}


# Reader-facing labels only
configuration_labels = {
    "Layer 1 composition":
        "Layer 1 — Mechanism composition",

    "Total Layer 3":
        "Layer 3 — Structural states and importance positions",

    "Full configuration":
        "Full task-affordance configuration",

    "Headline Layer 3 position":
        "Layer 3 — Importance positions",
}

hypothesis_labels = {
    "Layer 1 composition":
        "Automation- and augmentation-share associations "
        "are equal across sectors",

    "Total Layer 3":
        "Layer 3 associations are equal across sectors",

    "Full configuration":
        "All configuration associations are equal across sectors",

    "Headline Layer 3 position":
        "Importance-position associations are equal across sectors",
}


wald_rows = []

for test_name, restrictions in equality_tests.items():

    test_result = employment_sector_model.f_test(
        ", ".join(restrictions)
    )

    wald_rows.append({
        "test": test_name,
        "configuration_component":
            configuration_labels[test_name],
        "cross_sector_hypothesis":
            hypothesis_labels[test_name],
        "df_num": int(test_result.df_num),
        "df_denom": int(test_result.df_denom),
        "f_statistic": float(
            np.asarray(test_result.fvalue).squeeze()
        ),
        "p_value": float(
            np.asarray(test_result.pvalue).squeeze()
        ),
    })


# Raw analytical result retained
employment_sector_wald = pd.DataFrame(wald_rows)


# Reader-facing Main-table display
employment_sector_main = employment_sector_wald.copy()

employment_sector_main["Cluster-robust Wald F statistic"] = (
    "F("
    + employment_sector_main["df_num"].astype(str)
    + ", "
    + employment_sector_main["df_denom"].astype(str)
    + ") = "
    + employment_sector_main["f_statistic"].map("{:.3f}".format)
)

employment_sector_main["p value"] = (
    employment_sector_main["p_value"]
    .apply(
        lambda p:
            "< .001"
            if p < .001
            else f"{p:.4f}".lstrip("0")
    )
)

employment_sector_main = (
    employment_sector_main[
        [
            "configuration_component",
            "cross_sector_hypothesis",
            "Cluster-robust Wald F statistic",
            "p value",
        ]
    ]
    .rename(columns={
        "configuration_component":
            "Configuration component",
        "cross_sector_hypothesis":
            "Cross-sector hypothesis tested",
    })
)

In [ ]:
print("Main Table")
display(
    employment_sector_main.style
    .set_caption(
        "Table X. Joint Tests of Cross-Sector Heterogeneity in Task-Affordance Configuration Associations with Annualized Log Employment Change"
    )
    .hide(axis="index")
)

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 5: Sector-specific Layer 1 employment estimates
# Exact DEV Step 7E + supplement-facing presentation
# ------------------------------------------------------------------

layer1_sector_rows = []

for sector in ["Education", "Finance", "Healthcare", "Legal"]:
    for term in ["automation_share", "augmentation_share"]:

        parameter = f"C(sector)[{sector}]:{term}"

        ci_lower, ci_upper = (
            employment_sector_model.conf_int().loc[parameter]
        )

        layer1_sector_rows.append({
            "sector": sector,
            "term": term,
            "estimate": employment_sector_model.params[parameter],
            "cluster_standard_error":
                employment_sector_model.bse[parameter],
            "ci_95_lower": ci_lower,
            "ci_95_upper": ci_upper,
        })

employment_layer1_sector_estimates = pd.DataFrame(
    layer1_sector_rows
)

# Reader-facing labels only
employment_layer1_sector_estimates["term"] = (
    employment_layer1_sector_estimates["term"]
    .replace({
        "automation_share": "Automation share",
        "augmentation_share": "Augmentation share",
    })
)

employment_layer1_sector_estimates["95% CI"] = (
    "["
    + employment_layer1_sector_estimates["ci_95_lower"]
        .map("{:.4f}".format)
    + ", "
    + employment_layer1_sector_estimates["ci_95_upper"]
        .map("{:.4f}".format)
    + "]"
)

employment_layer1_sector_estimates = (
    employment_layer1_sector_estimates[
        [
            "sector",
            "term",
            "estimate",
            "cluster_standard_error",
            "95% CI",
        ]
    ]
    .rename(columns={
        "sector": "Sector",
        "term": "Configuration component",
        "estimate": "Estimate β",
        "cluster_standard_error": "Cluster-robust SE",
    })
)

In [ ]:
print("Supplementary Table")
display(
    employment_layer1_sector_estimates.style
    .set_caption(
        "Supplementary Table Sx. Sector-Specific Layer 1 "
        "Task-Affordance Composition Associations with "
        "Annualized Log Employment Change"
    )
    .format({
        "Estimate β": "{:.4f}",
        "Cluster-robust SE": "{:.4f}",
    })
    .hide(axis="index")
)

Mean-wage sector stack

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 6: Construct temporary mean-wage sector stack
# Exact DEV Step 7F — redundant display omitted
# ------------------------------------------------------------------

tmp_mean_wage_base = model_data.loc[
    model_data["sample_mean_wage"],
    [
        "soc6",
        "mean_wage_ann_log_change",
        *sector_model_terms,
        *sector_cols.values(),
    ]
].copy()

tmp_sector_mean_wage = tmp_mean_wage_base.melt(
    id_vars=[
        "soc6",
        "mean_wage_ann_log_change",
        *sector_model_terms,
    ],
    value_vars=list(sector_cols.values()),
    var_name="sector_column",
    value_name="sector_member",
)

tmp_sector_mean_wage = (
    tmp_sector_mean_wage.loc[
        tmp_sector_mean_wage["sector_member"]
    ]
    .drop(columns="sector_member")
    .assign(
        sector=lambda df:
            df["sector_column"]
            .str.replace("sector_", "", regex=False)
            .str.title()
    )
    .drop(columns="sector_column")
    .reset_index(drop=True)
)

In [ ]:
assert len(tmp_sector_mean_wage) == 1221
assert tmp_sector_mean_wage["soc6"].nunique() == 537
assert not tmp_sector_mean_wage.duplicated(
    ["soc6", "sector"]
).any()

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 7: Fit mean-wage sector model
# Exact DEV Step 7G
# ------------------------------------------------------------------

for sector in ["Education", "Finance", "Healthcare"]:
    column = f"limited_absent_{sector.lower()}"

    tmp_sector_mean_wage[column] = (
        (tmp_sector_mean_wage["sector"] == sector)
        & tmp_sector_mean_wage["limited_absent"].astype(bool)
    ).astype(int)

mean_wage_sector_formula = employment_sector_formula.replace(
    "employment_ann_log_change",
    "mean_wage_ann_log_change",
)

mean_wage_sector_model = smf.ols(
    formula=mean_wage_sector_formula,
    data=tmp_sector_mean_wage,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": tmp_sector_mean_wage["soc6"],
        "use_correction": True,
    },
    use_t=True,
)

tmp_mean_wage_design_audit = pd.DataFrame({
    "stacked_rows": [int(mean_wage_sector_model.nobs)],
    "soc6_clusters": [
        tmp_sector_mean_wage["soc6"].nunique()
    ],
    "estimated_parameters": [
        len(mean_wage_sector_model.params)
    ],
    "design_matrix_rank": [
        np.linalg.matrix_rank(
            mean_wage_sector_model.model.exog
        )
    ],
})

display(tmp_mean_wage_design_audit)

In [ ]:
assert len(mean_wage_sector_model.params) == 39
assert np.linalg.matrix_rank(
    mean_wage_sector_model.model.exog
) == 39

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 8: Mean-wage cross-sector equality tests
# Exact DEV Step 7H + reader-facing presentation
# ------------------------------------------------------------------

mean_wage_wald_rows = []

for test_name, restrictions in equality_tests.items():

    test_result = mean_wage_sector_model.f_test(
        ", ".join(restrictions)
    )

    mean_wage_wald_rows.append({
        "test": test_name,
        "configuration_component":
            configuration_labels[test_name],
        "cross_sector_hypothesis":
            hypothesis_labels[test_name],
        "df_num": int(test_result.df_num),
        "df_denom": int(test_result.df_denom),
        "f_statistic": float(
            np.asarray(test_result.fvalue).squeeze()
        ),
        "p_value": float(
            np.asarray(test_result.pvalue).squeeze()
        ),
    })

# Raw analytical result retained
mean_wage_sector_wald = pd.DataFrame(
    mean_wage_wald_rows
)

# Reader-facing Main-table display
mean_wage_sector_main = mean_wage_sector_wald.copy()

mean_wage_sector_main["Cluster-robust Wald F statistic"] = (
    "F("
    + mean_wage_sector_main["df_num"].astype(str)
    + ", "
    + mean_wage_sector_main["df_denom"].astype(str)
    + ") = "
    + mean_wage_sector_main["f_statistic"].map("{:.3f}".format)
)

mean_wage_sector_main["p value"] = (
    mean_wage_sector_main["p_value"]
    .apply(
        lambda p:
            "< .001"
            if p < .001
            else f"{p:.4f}".lstrip("0")
    )
)

mean_wage_sector_main = (
    mean_wage_sector_main[
        [
            "configuration_component",
            "cross_sector_hypothesis",
            "Cluster-robust Wald F statistic",
            "p value",
        ]
    ]
    .rename(columns={
        "configuration_component":
            "Configuration component",
        "cross_sector_hypothesis":
            "Cross-sector hypothesis tested",
    })
)

Do these relationships vary across the four sectors?

In [ ]:
print("Main Table")
display(
    mean_wage_sector_main.style
    .set_caption(
        "Table X. Joint Tests of Cross-Sector Heterogeneity "
        "in Task-Affordance Configuration Associations with "
        "Annualized Log Mean-Wage Change"
    )
    .hide(axis="index")
)

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 9: Sector-specific Layer 1 mean-wage estimates
# Exact DEV Step 7I + supplement-facing presentation
# ------------------------------------------------------------------

mean_wage_layer1_rows = []

for sector in ["Education", "Finance", "Healthcare", "Legal"]:
    for term in ["automation_share", "augmentation_share"]:

        parameter = f"C(sector)[{sector}]:{term}"

        ci_lower, ci_upper = (
            mean_wage_sector_model.conf_int().loc[parameter]
        )

        mean_wage_layer1_rows.append({
            "sector": sector,
            "term": term,
            "estimate": mean_wage_sector_model.params[parameter],
            "cluster_standard_error":
                mean_wage_sector_model.bse[parameter],
            "ci_95_lower": ci_lower,
            "ci_95_upper": ci_upper,
        })

mean_wage_layer1_sector_estimates = pd.DataFrame(
    mean_wage_layer1_rows
)

# Reader-facing labels only
mean_wage_layer1_sector_estimates["term"] = (
    mean_wage_layer1_sector_estimates["term"]
    .replace({
        "automation_share": "Automation share",
        "augmentation_share": "Augmentation share",
    })
)

mean_wage_layer1_sector_estimates["95% CI"] = (
    "["
    + mean_wage_layer1_sector_estimates["ci_95_lower"]
        .map("{:.4f}".format)
    + ", "
    + mean_wage_layer1_sector_estimates["ci_95_upper"]
        .map("{:.4f}".format)
    + "]"
)

mean_wage_layer1_sector_estimates = (
    mean_wage_layer1_sector_estimates[
        [
            "sector",
            "term",
            "estimate",
            "cluster_standard_error",
            "95% CI",
        ]
    ]
    .rename(columns={
        "sector": "Sector",
        "term": "Configuration component",
        "estimate": "Estimate β",
        "cluster_standard_error": "Cluster-robust SE",
    })
)

What do the sector-specific relationships actually look like?

In [ ]:
print("Supplementary Table")
display(
    mean_wage_layer1_sector_estimates.style
    .set_caption(
        "Supplementary Table Sx. Sector-Specific Layer 1 "
        "Task-Affordance Composition Associations with "
        "Annualized Log Mean-Wage Change"
    )
    .format({
        "Estimate β": "{:.4f}",
        "Cluster-robust SE": "{:.4f}",
    })
    .hide(axis="index")
)

### Median-wage sector stack

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 10: Construct temporary median-wage sector stack
# Exact DEV Step 7J — redundant display omitted
# ------------------------------------------------------------------

tmp_median_wage_base = model_data.loc[
    model_data["sample_median_wage"],
    [
        "soc6",
        "median_wage_ann_log_change",
        *sector_model_terms,
        *sector_cols.values(),
    ]
].copy()

tmp_sector_median_wage = tmp_median_wage_base.melt(
    id_vars=[
        "soc6",
        "median_wage_ann_log_change",
        *sector_model_terms,
    ],
    value_vars=list(sector_cols.values()),
    var_name="sector_column",
    value_name="sector_member",
)

tmp_sector_median_wage = (
    tmp_sector_median_wage.loc[
        tmp_sector_median_wage["sector_member"]
    ]
    .drop(columns="sector_member")
    .assign(
        sector=lambda df:
            df["sector_column"]
            .str.replace("sector_", "", regex=False)
            .str.title()
    )
    .drop(columns="sector_column")
    .reset_index(drop=True)
)

In [ ]:
assert len(tmp_sector_median_wage) == 1198
assert tmp_sector_median_wage["soc6"].nunique() == 526
assert not tmp_sector_median_wage.duplicated(
    ["soc6", "sector"]
).any()

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 11: Fit median-wage sector model
# Exact DEV Step 7K
# ------------------------------------------------------------------

for sector in ["Education", "Finance", "Healthcare"]:
    column = f"limited_absent_{sector.lower()}"

    tmp_sector_median_wage[column] = (
        (tmp_sector_median_wage["sector"] == sector)
        & tmp_sector_median_wage["limited_absent"].astype(bool)
    ).astype(int)

median_wage_sector_formula = employment_sector_formula.replace(
    "employment_ann_log_change",
    "median_wage_ann_log_change",
)

median_wage_sector_model = smf.ols(
    formula=median_wage_sector_formula,
    data=tmp_sector_median_wage,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": tmp_sector_median_wage["soc6"],
        "use_correction": True,
    },
    use_t=True,
)

tmp_median_wage_design_audit = pd.DataFrame({
    "stacked_rows": [int(median_wage_sector_model.nobs)],
    "soc6_clusters": [
        tmp_sector_median_wage["soc6"].nunique()
    ],
    "estimated_parameters": [
        len(median_wage_sector_model.params)
    ],
    "design_matrix_rank": [
        np.linalg.matrix_rank(
            median_wage_sector_model.model.exog
        )
    ],
})

display(tmp_median_wage_design_audit)

assert len(median_wage_sector_model.params) == 39
assert np.linalg.matrix_rank(
    median_wage_sector_model.model.exog
) == 39

In [ ]:
# ------------------------------------------------------------------
# RQ1c — Sector heterogeneity
# Step 12: Median-wage cross-sector equality tests
# ------------------------------------------------------------------

median_wage_wald_rows = []

for test_name, restrictions in equality_tests.items():

    test_result = median_wage_sector_model.f_test(
        ", ".join(restrictions)
    )

    median_wage_wald_rows.append({
        "test": test_name,
        "configuration_component":
            configuration_labels[test_name],
        "cross_sector_hypothesis":
            hypothesis_labels[test_name],
        "df_num": int(test_result.df_num),
        "df_denom": int(test_result.df_denom),
        "f_statistic": float(
            np.asarray(test_result.fvalue).squeeze()
        ),
        "p_value": float(
            np.asarray(test_result.pvalue).squeeze()
        ),
    })

# Raw analytical result retained
median_wage_sector_wald = pd.DataFrame(
    median_wage_wald_rows
)

# Reader-facing Main-table display
median_wage_sector_main = median_wage_sector_wald.copy()

median_wage_sector_main["Cluster-robust Wald F statistic"] = (
    "F("
    + median_wage_sector_main["df_num"].astype(str)
    + ", "
    + median_wage_sector_main["df_denom"].astype(str)
    + ") = "
    + median_wage_sector_main["f_statistic"].map("{:.3f}".format)
)

median_wage_sector_main["p value"] = (
    median_wage_sector_main["p_value"]
    .apply(
        lambda p:
            "< .001"
            if p < .001
            else f"{p:.4f}".lstrip("0")
    )
)

median_wage_sector_main = (
    median_wage_sector_main[
        [
            "configuration_component",
            "cross_sector_hypothesis",
            "Cluster-robust Wald F statistic",
            "p value",
        ]
    ]
    .rename(columns={
        "configuration_component":
            "Configuration component",
        "cross_sector_hypothesis":
            "Cross-sector hypothesis tested",
    })
)

In [ ]:
display(
    median_wage_sector_main.style
    .set_caption(
        "Joint Tests of Cross-Sector Heterogeneity "
    )
    .hide(axis="index")
)